<a href="https://colab.research.google.com/github/nika19du/Travel-Guide-AI-Integration/blob/main/Tour_Guide_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Exam - AI Integration for Developers

## 1. Install dependencies

In [1]:
!pip install openai chromadb pymupdf pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 41.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.

## 2. Imports

In [12]:
import os
import re
import json
import fitz
import chromadb

from pathlib import Path
from typing import Type, TypeVar, List, Literal, Dict, Any

from openai import OpenAI
from pydantic import BaseModel, Field, ConfigDict
from rich.console import Console
from rich.panel import Panel
from rich.table import Table
from rich.text import Text

## 3. OpenAI client

In [7]:
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")
)

## 4. Models

In [8]:
class IntentAnalysis(BaseModel):
    prompt: str = Field(
        description="The complete factual question to answer."
    )
    format: Literal["text", "image", "audio"] = Field(
        description="Requested output format."
    )

class TravelGuideItem(BaseModel):
    title: str = Field(
        description = (
            "Short title of the retrieved recommendation, place, "
            "period, transport option, accommodation, or other result."
        )
    )

    description: str = Field(
        description= (
            "Factual description based only on the retrieved context."
        )
    )

class DestinationTravelAnswer(BaseModel):
    destination: str = Field(
        description=(
            "The unique destination described by this answer. "
            "There must be only one answer object per destination."
        )
    )

    summary: str = Field(
        description=(
            "A complete direct answer to the user's question for this "
            "destination. Combine all relevant aspects in one summary."
        )
    )

    items: List[TravelGuideItem] = Field(
        default_factory=list,
        description=(
            "All relevant facts for this destination. For example, if the "
            "source provides both the best and cheapest travel periods, "
            "include both as separate items inside this single destination "
            "object instead of creating duplicate destination objects."
        )
    )

    source_files: List[str] = Field(
        description="PDF filenames that directly support the answer."
    )

    source_pages: List[int] = Field(
        description=(
            "Only the page numbers that directly support the answer."
        )
    )

    missing_information: List[str] = Field(
        default_factory=list,
        description=(
            "Information requested by the user but not explicitly available "
            "in the retrieved context."
        )
    )

class TravelGuideAnswer(BaseModel):
    destinations: List[DestinationTravelAnswer] =  Field(
        description=(
            "One answer object per unique destination. "
            "Never create multiple objects for the same destination. "
            "Combine all relevant information for a destination into "
            "a single object and place the individual facts in items."
        )
    )

class AskAIResult(BaseModel):
    intent: IntentAnalysis
    answer: TravelGuideAnswer

##5. PDF Utils

In [26]:
import re
from pathlib import Path

import fitz

SECTION_HEADINGS = (
    "Best Time to Visit",
    "Worst Time to Visit",
    "Cheapest Time to Visit"
)

def extract_pdf_pages(pdf_path: str) -> list[dict]:
    path = Path(pdf_path)
    path = Path(pdf_path)

    if not path.exists():
        raise FileNotFoundError(f'PDF not found: {pdf_path}')

    document = fitz.open(path)

    try:
        pages = []
        for page_index,page in enumerate(document):
            text = page.get_text("text").strip()

            if text:
                pages.append({'page': page_index+1, 'text': text})

        return pages
    finally:
        document.close()

def split_text(
    text: str,
    chunk_size: int = 1200,
    chunk_overlap: int = 200
) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be greater than zero.")

    if chunk_overlap < 0:
        raise ValueError("chunk_overlap cannot be negative.")

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller than chunk_size."
        )

    if not text or not text.strip():
        return []

    sections = split_by_headings(text)

    chunks = []

    for section in sections:
        chunks.extend(
            split_by_size(
                text = section,
                chunk_size = chunk_size,
                chunk_overlap = chunk_overlap
            )
        )

    return chunks

def split_by_headings(text: str) -> list[str]:
    headings_pattern = "|".join(
        re.escape(heading) for heading in SECTION_HEADINGS
    )

    pattern = rf"(?=^(?:{headings_pattern})\s*$)"

    sections = re.split(
        pattern,
        text,
        flags=re.IGNORECASE | re.MULTILINE
    )

    return [
        section.strip()
        for section in sections
        if section.strip()
    ]

def split_by_size(
    text: str,
    chunk_size: int,
    chunk_overlap: int
) -> list[str]:
    if len(text) <= chunk_size:
        return [text]

    chunks = []
    step = chunk_size - chunk_overlap

    for start in range(0, len(text), step):
        chunk = text [
            start: start + chunk_size
        ].strip()

        if chunk:
            chunks.append(chunk)
    return chunks

## 6. Vector Store

In [27]:
class VectorStoreManager:
    def __init__(
        self,
        db_path,
        collection_name: str = "travel_guides",
        embedding_model="text-embedding-3-small",
    ):
        self.db_path = Path(db_path)
        self.collection_name = collection_name
        self.embedding_model = embedding_model

        self.chroma_client = chromadb.PersistentClient(
            path=str(self.db_path)
        )

        self.collection = self.chroma_client.get_or_create_collection(
            name=self.collection_name
        )

    def add_pdf(self, pdf_path: str) -> None:
        source = Path(pdf_path).name
        pages = extract_pdf_pages(pdf_path)

        ids = []
        documents = []
        embeddings = []
        metadatas = []

        for page_data in pages:
            page_number = page_data["page"]
            chunks = split_text(page_data["text"])

            for chunk_index, chunk in enumerate(chunks):
                chunk_id = (
                    f"{source}_page{page_number}_{chunk_index}"
                )

                embedding = self._create_embedding(chunk)

                ids.append(chunk_id)
                documents.append(chunk)
                embeddings.append(embedding)

                metadatas.append({
                    "source": source,
                    "page": page_number,
                    "chunk_index": chunk_index
                })

        if not ids:
            print(f"No chunks found in {source}.")
            return

        self.collection.upsert(
            ids=ids,
            documents=documents,
            embeddings=embeddings,
            metadatas=metadatas
        )

        print(f"Indexed {source} ({len(ids)} chunks)")

    def count_chunks(self):
        return self.collection.count()

    def _create_embedding(self, text):
        response = client.embeddings.create(
            model=self.embedding_model,
            input=text
        )

        return response.data[0].embedding

    def search(
        self,
        query: str,
        source: str | None = None,
        top_k: int = 5,
    ) -> list[dict]:
        if not query or not query.strip():
            raise ValueError("Search query cannot be empty.")

        if top_k < 1:
            raise ValueError(
                "top_k must be greater than zero."
            )

        normalized_query = query.strip().lower()

        query_embedding = self._create_embedding(
            query.strip()
        )

        query_arguments = {
            "query_embeddings": [query_embedding],
            "n_results": max(top_k, 5),
            "include": [
                "documents",
                "metadatas",
                "distances"
            ]
        }

        if source is not None:
            query_arguments["where"] = {
                "source": source
            }

        results = self.collection.query(
            **query_arguments
        )

        chunks = []

        result_ids = results.get("ids", [[]])[0]
        documents = results.get("documents", [[]])[0]
        metadatas = results.get("metadatas", [[]])[0]
        distances = results.get("distances", [[]])[0]

        for chunk_id, document, metadata, distance in zip(
            result_ids,
            documents,
            metadatas,
            distances
        ):
            normalized_document = document.strip().lower()

            exact_heading_match = (
                normalized_document.startswith(normalized_query)
            )

            adjusted_distance = distance

            if exact_heading_match:
                adjusted_distance -= 1.0

            chunks.append({
                "id": chunk_id,
                "text": document,
                "source": metadata.get("source"),
                "page": metadata.get("page"),
                "chunk_index": metadata.get("chunk_index"),
                "distance": distance,
                "_adjusted_distance": adjusted_distance
            })

        chunks.sort(key = lambda chunk: chunk["_adjusted_distance"])

        selected_chunks = chunks[:top_k]

        for chunk in selected_chunks:
            chunk.pop("_adjusted_distance", None)

        return selected_chunks

    def list_available_guides(self):
        results = self.collection.get(
            include=["metadatas"]
        )

        metadatas = results.get("metadatas", [])

        sources = set()

        for metadata in metadatas:
            if not metadata:
                continue

            source = metadata.get("source")

            if source:
                sources.add(source)

        return sorted(sources)

## 7. Tools

In [16]:
SOURCE_MAPPING = {
    "Prague": "prague_tour_guide.pdf",
    "Malaga": "malaga_tour_guide.pdf",
}

class SearchTravelGuidesParams(BaseModel):
    model_config = ConfigDict(extra="forbid")

    query: str = Field(
        description=(
            "A concise semantic search query describing "
            "the requested information."
        )
    )
    destination: str | None = Field(
        description = (
            "Destination filter, for example Prague or Malaga. "
            "Use null when no destination filter is required."
        )
    )
    top_k: int = Field(
        ge = 3,
        le = 10,
        description=(
            "Number of retrieved chunks. Use 5 by default "
            "and never fewer than 3."
        )
    )

class ListAvailableTravelGuidesParams(BaseModel):
    model_config = ConfigDict(extra="forbid")

TRAVEL_GUIDE_TOOLS = [
    {
        "type": "function",
        "name": "search_travel_guides",
        "description": (
            "Search the indexed travel guides for information "
            "about destinations, weather, prices, attractions, "
            "transport, accommodation and recommended travel periods."
        ),
        "parameters": SearchTravelGuidesParams.model_json_schema(),
        "strict": True,
    },
    {
        "type": "function",
        "name": "list_available_guides",
        "description": (
            "List the destinations and travel guides "
            "available in the vector database."
        ),
        "parameters": ListAvailableTravelGuidesParams.model_json_schema(),
        "strict": True,
    },
]


def search_travel_guides(
    vector_store: VectorStoreManager,
    query: str,
    destination: str | None,
    top_k = 5
):
    """
    Search the indexed travel guides using semantic search.
    Args:
        vector_store: ChromaDB vector store.
        query: User search query.
        source: Optional PDF source filter.
        top_k: Maximum number of chunks to return.
    """

    source = SOURCE_MAPPING.get(destination)

    if destination is not None and source is None:
        return {
            "success": False,
            "error": f"No indexed guide found for {destination}"
        }

    safe_top_k = max(3, min(top_k, 10))

    results = vector_store.search(
        query = query,
        source = source,
        top_k = safe_top_k)

    return {
        "success": True,
        "data": {
            "query": query,
            "destination": destination,
            "source_filter": source,
            "result_count": len(results),
            "results": results
        }
    }


def list_available_guides(vector_store: VectorStoreManager):
    guides = vector_store.list_available_guides()

    return {
        "success": True,
        "data": {
            "guide_count": len(guides),
            "indexed_chunk_count": vector_store.count_chunks(),
            "guides": guides
        }
    }

def execute_tool(
    tool_name: str,
    arguments: dict,
    vector_store: VectorStoreManager
):
    if tool_name == "search_travel_guides":
        params = SearchTravelGuidesParams.model_validate(arguments)

        return search_travel_guides(
            vector_store = vector_store,
            query = params.query,
            destination = params.destination,
            top_k = params.top_k
        )

    if tool_name == "list_available_guides":
        ListAvailableTravelGuidesParams.model_validate(arguments)
        return list_available_guides(vector_store)

    raise ValueError(f"Tool '{tool_name}' not found.")

##8. Prompts

In [17]:
RETRIEVAL_SYSTEM_PROMPT = """
You are a retrieval-grounded travel assistant.

Use ONLY facts explicitly stated in the retrieved PDF context.

Rules:
1. Answer only the exact question asked by the user.
2. Return exactly one DestinationTravelAnswer per unique destination.
3. Never create multiple objects for the same destination.
4. Include only facts that are necessary to answer the user's question.
5. Ignore other sections from the same retrieved chunk when they do not
   directly answer the question.

6. If the user asks for the best time to visit:
   - include only information from the "Best Time to Visit" section;
   - include only recommended or favourable periods;
   - do not include cheapest periods;
   - do not include worst periods;
   - do not include information from the "Worst Time to Visit" section;
   - do not include monthly temperatures;
   - do not include rainy-day statistics;
   - do not mention unfavourable months in the summary or items;
   - unless the user explicitly asks for them.

7. If the user asks for the cheapest time:
   - include only information from the "Cheapest Time to Visit" section;
   - include only price-related periods;
   - do not include best-weather periods;
   - do not include worst periods;
   - unless explicitly requested.

8. If the user asks for attractions:
   - include only attractions and directly relevant descriptions.

9. Do not treat every fact in a retrieved chunk as relevant.
10. Preserve complete ranges and alternatives exactly as stated.
11. Do not infer facts from general travel knowledge.
12. Use only source filenames and page numbers from the retrieved context.
13. Include only pages that directly support the returned answer.
14. Put information in missing_information only when the requested
    information is absent from the retrieved context.
15. The destinations array must contain only actual travel destinations
    that exist in the travel guides.
16. Never create synthetic entries such as "Comparison", "Summary",
    or "Overall".
17. Before returning the final answer, remove every item that does not
    directly answer the user's exact question.
"""

INTENT_SYSTEM_PROMPT = """
Analyse the user request.

Extract the complete factual question that must be answered
from the PDF documents.

Remove only instructions regarding the requested output format.

Preserve:
- destination names,
- comparison criteria,
- conditions and qualifiers,
- references to the guides.

When the user does not explicitly request image or audio,
use text as the output format.
"""

##9. AI Logic using RAG service, ask_ai

In [55]:
T = TypeVar("T", bound=BaseModel)

DEBUG = False
MAX_TOOL_ITERATIONS = 8

def generate_structured_response(
    instructions: str,
    user_input: str,
    response_model: Type[T]
) -> T:
    response = client.responses.parse(
        model="gpt-4o-mini",
        instructions=instructions,
        input=user_input,
        text_format=response_model
    )

    parsed_result = response.output_parsed

    if parsed_result is None:
        raise ValueError(
            f"The model did not return a valid "
            f"{response_model.__name__} response."
        )

    return parsed_result


def retrieve_information_with_store(
    prompt: str,
    vector_store: VectorStoreManager
) -> TravelGuideAnswer:

    instructions = (
        f"{RETRIEVAL_SYSTEM_PROMPT}\n\n"
        "You have access to indexed PDF travel guides.\n\n"
        "Available destinations:\n"
        "- Prague\n"
        "- Malaga\n\n"
        "Tool usage rules:\n"
        "1. If the question mentions Prague, call search_travel_guides "
        "with destination='Prague'.\n"
        "2. If the question mentions Malaga, call search_travel_guides "
        "with destination='Malaga'.\n"
        "3. If the user explicitly asks to compare Prague and Malaga, "
        "perform a separate search for each destination.\n"
        "4. Do not search an unrequested destination.\n"
        "5. Use list_available_guides only when you need to discover "
        "which guides are available.\n"
        "6. Base the answer only on retrieved chunks.\n"
        "7. For search_travel_guides, use top_k=5 by default "
        "and never use a value lower than 3.\n"
    )

    response = client.responses.parse(
        model = "gpt-4o-mini",
        instructions = instructions,
        input = prompt,
        tools = TRAVEL_GUIDE_TOOLS,
        text_format = TravelGuideAnswer
    )

    for iteration in range(MAX_TOOL_ITERATIONS):
        function_calls = [
            output_item
            for output_item in response.output
            if output_item.type == "function_call"
        ]

        # No tool calls means the model has produced its final answer.
        if not function_calls:
            parsed_answer = response.output_parsed

            if parsed_answer is None:
                raise ValueError(
                    "The model did not return a valid "
                    "TravelGuideAnswer."
                )

            invalid_destinations = {
                "comparison",
                "summary",
                "overall"
            }

            parsed_answer.destinations = [
                destination
                for destination in parsed_answer.destinations
                if destination.destination.strip().lower()
                not in invalid_destinations
            ]

            return parsed_answer

        tool_outputs = []

        for function_call in function_calls:
            if DEBUG:
                print(
                    f"\nTool call: {function_call.name}"
                )
                print(
                    f"Arguments: {function_call.arguments}"
                )

            tool_result = run_function_call(
                function_call=function_call,
                vector_store=vector_store
            )

            if DEBUG:
                print("Tool result:")
                print(
                    json.dumps(
                        tool_result,
                        indent=2,
                        ensure_ascii=False
                    )
                )

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": function_call.call_id,
                "output": json.dumps(
                    tool_result,
                    ensure_ascii=False
                )
            })

        # Send the locally executed tool results back to the model.
        response = client.responses.parse(
            model = "gpt-4o-mini",
            instructions = instructions,
            previous_response_id = response.id,
            input = tool_outputs,
            tools = TRAVEL_GUIDE_TOOLS,
            text_format = TravelGuideAnswer
        )

    raise RuntimeError(
        "The maximum number of tool-call iterations "
        "was exceeded."
    )


def run_function_call(
    function_call,
    vector_store: VectorStoreManager
):
    """
    Converts the JSON tool arguments into a Python dictionary
    and executes the requested local function.
    """

    try:
        arguments = json.loads(function_call.arguments)
    except json.JSONDecodeError as error:
        return {
            "success": False,
            "error": "Invalid tool arguments.",
            "details": str(error)
        }

    try:
        return execute_tool(
            vector_store=vector_store,
            tool_name=function_call.name,
            arguments=arguments
        )
    except Exception as error:
        return {
            "success": False,
            "error": (
                f"Tool '{function_call.name}' failed."
            ),
            "details": str(error)
        }


def ask_ai_with_store(
    question: str,
    vector_store: VectorStoreManager
) -> AskAIResult:
    print(
        "\n"
        "--------------------------------------------------"
    )
    print(f'User Question: "{question}"')

    if not question or not question.strip():
        raise ValueError(
            "The user question cannot be empty."
        )

    # Step 1:
    # Analyze the user's original question.
    parsed_intent = generate_structured_response(
        instructions=INTENT_SYSTEM_PROMPT,
        user_input=question.strip(),
        response_model=IntentAnalysis
    )

    extracted_prompt = parsed_intent.prompt
    requested_format = parsed_intent.format

    print(f'Extracted Prompt: "{extracted_prompt}"')
    print(
        f'Detected Format: '
        f'"{requested_format.upper()}"'
    )

    # Step 2:
    # Let the model choose and execute the retrieval tools.
    travel_answer = retrieve_information_with_store(
        prompt=extracted_prompt,
        vector_store=vector_store
    )

    return AskAIResult(
        intent=parsed_intent,
        answer=travel_answer
    )

In [56]:
DB_PATH = "/content/chroma_db"
GUIDES_DIRECTORY = "/content/travel_guides"


vector_store = VectorStoreManager(
    db_path=DB_PATH,
    collection_name="travel_guides",
    embedding_model="text-embedding-3-small"
)


## Wrapper for retrieve_information(prompt) & ask_ai(question)

In [57]:
def retrieve_information(prompt: str) -> str:
    """
    Required project function.

    Retrieves information from the indexed PDF guides
    and returns it as a string.
    """

    answer = retrieve_information_with_store(
        prompt=prompt,
        vector_store=vector_store
    )

    return answer.model_dump_json(
        indent=2
    )


def ask_ai(question: str) -> AskAIResult:
    """
    Required project function.

    Accepts only the user's question. The configured
    vector store is supplied internally.
    """

    return ask_ai_with_store(
        question=question,
        vector_store=vector_store
    )

## 10. Console

In [41]:
console = Console()

def print_comparison_table(answer: TravelGuideAnswer) -> None:
    """
    Prints a compact comparison table when two or more
    destinations are included in the answer.
    """

    if len(answer.destinations) < 2:
        return

    table = Table(
        title="Travel Comparison",
        title_style="bold cyan",
        header_style="bold",
        show_header=True,
        show_lines=True,
        expand=True,
        padding=(0, 1),
    )

    table.add_column(
        "Destination",
        style="bold cyan",
        no_wrap=True,
        min_width=12,
        max_width=18,
    )

    table.add_column(
        "Category",
        style="bold",
        min_width=18,
        ratio=1,
    )

    table.add_column(
        "Recommendation",
        ratio=3,
    )

    for destination in answer.destinations:
        if destination.items:
            first_item = True

            for item in destination.items:
                table.add_row(
                    (
                        destination.destination
                        if first_item
                        else ""
                    ),
                    item.title,
                    item.description,
                )

                first_item = False
        else:
            table.add_row(
                destination.destination,
                "Summary",
                destination.summary,
            )

    console.print()
    console.print(table)
    console.print()


def print_destination_details(answer: TravelGuideAnswer) -> None:
    """
    Prints detailed information for every destination.
    """

    for destination in answer.destinations:
        console.rule(
            f"[bold cyan]{destination.destination}[/bold cyan]"
        )

        console.print()

        summary_panel = Panel(
            destination.summary,
            title="Summary",
            title_align="left",
            border_style="cyan",
            padding=(1, 2),
        )

        console.print(summary_panel)

        if destination.items:
            console.print()

            information_table = Table(
                header_style="bold",
                show_header=True,
                show_lines=False,
                expand=True,
                padding=(0, 1),
                box=None,
            )

            information_table.add_column(
                "Category",
                style="bold cyan",
                min_width=20,
                max_width=28,
            )

            information_table.add_column(
                "Details",
                ratio=3,
            )

            for item in destination.items:
                information_table.add_row(
                    item.title,
                    item.description,
                )

            console.print(information_table)

        if destination.missing_information:
            console.print()

            missing_text = Text()

            for index, missing_item in enumerate(
                destination.missing_information,
                start=1,
            ):
                missing_text.append(
                    f"{index}. ",
                    style="bold yellow",
                )
                missing_text.append(
                    f"{missing_item}\n"
                )

            console.print(
                Panel(
                    missing_text,
                    title="Missing Information",
                    title_align="left",
                    border_style="yellow",
                    padding=(1, 2),
                )
            )

        console.print()


def print_sources(answer: TravelGuideAnswer) -> None:
    """
    Prints source files and pages used for each
    destination.
    """

    destinations_with_sources = [
        destination
        for destination in answer.destinations
        if (
            destination.source_files
            or destination.source_pages
        )
    ]

    if not destinations_with_sources:
        return

    console.rule("[bold]Sources[/bold]")
    console.print()

    sources_table = Table(
        show_header=True,
        header_style="bold",
        show_lines=False,
        box=None,
        padding=(0, 1),
    )

    sources_table.add_column(
        "Destination",
        style="bold cyan",
        no_wrap=True,
        min_width=12,
    )

    sources_table.add_column(
        "Source file",
        min_width=24,
    )

    sources_table.add_column(
        "Pages",
        justify="right",
        no_wrap=True,
    )

    for destination in destinations_with_sources:
        source_files = ", ".join(
            destination.source_files
        )

        source_pages = ", ".join(
            str(page)
            for page in destination.source_pages
        )

        sources_table.add_row(
            destination.destination,
            source_files or "-",
            source_pages or "-",
        )

    console.print(sources_table)


def print_result_summary(answer: TravelGuideAnswer) -> None:
    """
    Prints a compact footer with result statistics.
    """

    destination_count = len(answer.destinations)

    label = (
        "destination"
        if destination_count == 1
        else "destinations"
    )

    console.print()
    console.print(
        Text.assemble(
            ("Analysed ", "dim"),
            (
                str(destination_count),
                "bold cyan",
            ),
            (
                f" {label}",
                "dim",
            ),
        )
    )


def print_travel_answer(answer: TravelGuideAnswer) -> None:
    if not answer.destinations:
        console.print(
            Panel(
                "No destination information was found.",
                border_style="yellow",
            )
        )
        return

    is_comparison = len(answer.destinations) >= 2

    if is_comparison:
        print_comparison_table(answer)
    else:
        print_destination_details(answer)

    print_sources(answer)
    print_result_summary(answer)


def print_ai_result(result: AskAIResult) -> None:
    """
    Prints the AI result according to the requested
    output format.
    """

    requested_format = result.intent.format.strip().lower()

    if requested_format == "text":
        print_travel_answer(result.answer)
        return

    if requested_format == "audio":
        console.print(
            Panel(
                "Audio output is not implemented yet.",
                border_style="yellow",
            )
        )
        return

    if requested_format == "image":
        console.print(
            Panel(
                "Image output is not implemented yet.",
                border_style="yellow",
            )
        )
        return

    console.print(
        Panel(
            (
                "Unsupported output format: "
                f"{requested_format}"
            ),
            border_style="yellow",
        )
    )

## 11. Main

In [59]:
def index_guides_if_needed(vector_store: VectorStoreManager):
    """
    Indexes the PDF files only when the ChromaDB
    collection is empty.
    """

    existing_chunks = vector_store.count_chunks()

    if existing_chunks > 0:
        print(
            f"Vector store already contains "
            f"{existing_chunks} chunks."
        )
        return

    guides_directory = Path(GUIDES_DIRECTORY)

    if not guides_directory.exists():
        raise FileNotFoundError(
            f"Travel guide directory does not exist: "
            f"{guides_directory.resolve()}"
        )

    pdf_files = sorted(
        guides_directory.glob("*.pdf")
    )

    if not pdf_files:
        raise FileNotFoundError(
            f"No PDF files were found in: "
            f"{guides_directory.resolve()}"
        )

    print(
        f"Found {len(pdf_files)} PDF guide(s)."
    )

    for pdf_file in pdf_files:
        vector_store.add_pdf(
            str(pdf_file)
        )

    print(
        f"Total indexed chunks: "
        f"{vector_store.count_chunks()}"
    )

def run_cli():
    print("\nTravel Guide AI Assistant")
    print(
        "Type 'exit' or 'quit' to close the application."
    )

    while True:
        question = input(
            "\nAsk a travel question: "
        ).strip()

        if question.lower() in {
            "exit",
            "quit"
        }:
            print("Goodbye!")
            break

        if not question:
            print("Please enter a question.")
            continue

        try:
            result = ask_ai(question)

            print_ai_result(result)

        except Exception as error:
            print(f"\nError: {error}")

def main():
    index_guides_if_needed(vector_store)

    available_guides = (
        vector_store.list_available_guides()
    )

    print("\nAvailable guides:")

    for guide in available_guides:
        print(f"- {guide}")

    run_cli()
if __name__ == "__main__":
    main()

Vector store already contains 229 chunks.

Available guides:
- malaga_tour_guide.pdf
- prague_tour_guide.pdf

Travel Guide AI Assistant
Type 'exit' or 'quit' to close the application.

Ask a travel question: How do I get from Malaga Airport to the city centre?

--------------------------------------------------
User Question: "How do I get from Malaga Airport to the city centre?"
Extracted Prompt: "How do I get from Malaga Airport to the city centre?"
Detected Format: "TEXT"


───────────────────────────────────────────────────── Malaga ──────────────────────────────────────────────────────

╭─ Summary ───────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  To get from Malaga Airport (AGP) to the city center, travelers have several transport options. The train       │
│  service takes about 12 minutes and costs around €1.80, running approximately every 20 minutes. Alternatively,  │
│  a taxi will cost between €20-30 and typically takes around 15 minutes. There is also a bus service that takes  │
│  approximately 20 minutes and costs about €4.                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Category              Details                                                                                     
 Train                 Takes about 12 minutes; costs €1.80; runs approximately every 20 minutes.                   
 Taxi                  Costs between €20-30; travel time is around 15 minutes.                                     
 Bus                   Takes approximately 20 minutes; costs about €4.

───────────────────────────────────────────────────── Sources ─────────────────────────────────────────────────────

...

                                                 Travel Comparison                                                 
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Destination  ┃ Category                ┃ Recommendation                                                         ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Prague       │ Best Weather Months     │ Late spring to summer (May to September) is ideal, especially June and │
│              │                         │ September.                                                             │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ Winter Magic Season     │ The winter holiday season turns Prague into a stunning wonderland.     │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│ Malaga       │ Best Beach Months       │ Mid-June to September is perfect for beach activities.                 │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ Best Sightseeing Months │ April-May and September-October are ideal for sightseeing.             │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│ prague       │ Summary                 │ Best time to visit is from late spring to summer, especially June and  │
│              │                         │ September. Also, winter holiday season brings a magical                │
│              │                         │ transformation.                                                        │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│ malaga       │ Summary                 │ Best time for beach life is from mid-June to September; for            │
│              │                         │ sightseeing, consider April-May and September-October.                 │
└──────────────┴─────────────────────────┴────────────────────────────────────────────────────────────────────────┘

───────────────────────────────────────────────────── Sources ─────────────────────────────────────────────────────

 Destination   Source file               Pages 
 Prague        prague_tour_guide.pdf        60 
 Malaga        malaga_tour_guide.pdf        49

Analysed 4 destinations


Ask a travel question: exit
Goodbye!


## 12. Test Cases

In [60]:
# ==========================
# Test Cases
# ==========================

test_questions = [
    "What are the top attractions in Prague?",
    "When is the best time to visit Prague?",
    "Which destination is cheaper, Prague or Malaga?",
    "Compare public transport in Prague and Malaga.",
    #"Create an image showing the top attractions in Prague.",
    #"Create an image of the beaches in Malaga.",
    #"Give me an audio summary of Prague.",
    #"Tell me about Malaga public transport as audio."
]

for index, question in enumerate(test_questions, start=1):
    console.rule(f"[bold cyan]Test Case {index}[/bold cyan]")

    try:
        result = ask_ai(question)
        print_ai_result(result)
    except Exception as error:
        console.print(f"[red]Error:[/red] {error}")

─────────────────────────────────────────────────── Test Case 1 ───────────────────────────────────────────────────


--------------------------------------------------
User Question: "What are the top attractions in Prague?"
Extracted Prompt: "What are the top attractions in Prague?"
Detected Format: "TEXT"


───────────────────────────────────────────────────── Prague ──────────────────────────────────────────────────────

╭─ Summary ───────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  Prague is known for its enchanting Gothic and Baroque architecture and rich history. Key attractions include   │
│  Prague Castle, St. Vitus Cathedral, Charles Bridge, the Astronomical Clock, and Wenceslas Square, among        │
│  others.                                                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

 Category              Details                                                                                     
 Prague Castle         One of the city's most important attractions, located in Hradčany.                          
 St. Vitus Cathedral   A significant landmark located within the grounds of Prague Castle.                         
 Charles Bridge        A historic bridge adorned with statues, connecting the Old Town with the Lesser Town.       
 Astronomical Clock    An iconic medieval clock located in the Old Town Square.                                    
 Wenceslas Square      A vibrant square that is a cultural and commercial hub of the Czech Republic.

...


--------------------------------------------------
User Question: "Compare public transport in Prague and Malaga."
Extracted Prompt: "Compare public transport in Prague and Malaga."
Detected Format: "TEXT"


                                                 Travel Comparison                                                 
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Destination  ┃ Category                ┃ Recommendation                                                         ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Prague       │ Metro                   │ Operates daily from 5:00 AM to midnight, with tickets at 30 CZK        │
│              │                         │ (€1.20) for 30 minutes.                                                │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ Bus                     │ Daytime service runs from 4:30 AM to midnight, night lines from 12:30  │
│              │                         │ AM to 4:30 AM. Tickets are 30 CZK (€1.20) for 30 minutes.              │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ Tram                    │ Runs from 4:30 AM to midnight, with tickets priced the same as buses.  │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ Funicular to Petřín     │ Operates daily from 8:00 AM to 11:00 PM, costing 60 CZK (€2.40).       │
│              │ Hill                    │                                                                        │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│ Malaga       │ Local Buses             │ Operates from 6:30 AM to 11:00 PM, with tickets priced at €1.40.       │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ Metro                   │ Runs Mon-Thu from 6:30 AM to 11:00 PM; single ticket costs €1.35.      │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ 10-Ride Bus Pass        │ Available for €8.40, making each ride approximately €0.82.             │
├──────────────┼─────────────────────────┼────────────────────────────────────────────────────────────────────────┤
│              │ Train from Airport      │ Line C1 to city center costs €1.80, approximately every 20 minutes.    │
└──────────────┴─────────────────────────┴────────────────────────────────────────────────────────────────────────┘

───────────────────────────────────────────────────── Sources ─────────────────────────────────────────────────────

 Destination   Source file                Pages 
 Prague        prague_tour_guide.pdf     41, 42 
 Malaga        malaga_tour_guide.pdf     32, 30

Analysed 2 destinations

## Unit Tests

---

